# Obtain Model Training Data from SEC EDGAR

This notebook documents the data acquisition process used to build the EDGAR-based financial statement dataset for the corporate credit risk model.

The workflow is intentionally split into safe, explicit steps:

1. Set project paths and execution flags.
2. Sanity-check EDGAR access.
3. Build or refresh the SEC ticker / CIK universe.
4. Enrich companies with SIC and industry classification.
5. Download raw EDGAR company facts.
6. Convert raw CSV checkpoints into parquet chunks.
7. Query parquet data with DuckDB.
8. Build the model base table.
9. Audit concept coverage.
10. Plan incremental refreshes for new tickers and new fiscal years.

Large EDGAR downloads are disabled by default to avoid accidental multi-gigabyte downloads. Enable the relevant `RUN_*` flags only when intentionally refreshing the dataset.


## 0. Imports

Keep imports in one place. This notebook uses `edgartools`, pandas, DuckDB, and parquet files as the local analytical layer.


In [1]:
from pathlib import Path
from datetime import date

# from pytickersymbols import PyTickerSymbols
from edgar import *
import nest_asyncio
nest_asyncio.apply()

import glob
import os
import pandas as pd
from tqdm import tqdm
import requests
from io import StringIO
import duckdb


## 1. Notebook configuration and project paths

The notebook lives inside `notebooks/`, so the repository root is one level above the current working directory.

All generated data is written below the root-level `data/` directory. This keeps the repository clean and prevents EDGAR outputs from being scattered across the project root.


In [2]:
pd.set_option("display.max_rows", 400)
pd.set_option("display.max_colwidth", 120)

# This notebook is expected to be run from the notebooks/ directory.
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"

RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
PARQUET_DIR = DATA_DIR / "raw_financial_facts_parquet"
DB_DIR = DATA_DIR / "duckdb"

for folder in [DATA_DIR, RAW_DIR, INTERIM_DIR, PROCESSED_DIR, PARQUET_DIR, DB_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

FUNDAMENTAL_UNIVERSE_PATH = RAW_DIR / "fundamental_universe.csv"
TICKER_CIK_SIC_PATH = INTERIM_DIR / "fundamental_universe_ticker_cik_sic.csv"
TICKER_SIC_INDUSTRY_PATH = PROCESSED_DIR / "03_fundamental_universe_ticker_sic_industry.csv"
RAW_FACTS_CSV_PATH = RAW_DIR / "raw_financial_facts.csv"
DUCKDB_PATH = DB_DIR / "financials.duckdb"

print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")


Project root: D:\users\kamen.dimitrov\desktop\SOFTUNI\AI_and_ML_upskill_program\machine_learning\08_final_project_3
Data directory: D:\users\kamen.dimitrov\desktop\SOFTUNI\AI_and_ML_upskill_program\machine_learning\08_final_project_3\data


## 2. Execution flags

The heavy EDGAR operations are guarded by explicit flags. Leave them as `False` when reviewing, presenting, or running the notebook casually.

Use `MAX_TICKERS_FOR_DOWNLOAD` for small test runs. A full refresh requires both `RUN_RAW_EDGAR_FACT_DOWNLOAD = True` and `CONFIRM_FULL_EDGAR_DOWNLOAD = True`.


In [3]:
RUN_EDGAR_SANITY_CHECK = False
RUN_TICKER_UNIVERSE_BUILD = False
RUN_SIC_TABLE_DOWNLOAD = False
RUN_SIC_ENRICHMENT = False
RUN_RAW_EDGAR_FACT_DOWNLOAD = False
RUN_CSV_TO_PARQUET = False
RUN_INCREMENTAL_REFRESH_AUDIT = False

MAX_TICKERS_FOR_DOWNLOAD = 10  # Use a small integer for testing; set to None only for full refresh.
CONFIRM_FULL_EDGAR_DOWNLOAD = False

TARGET_MIN_FISCAL_YEAR = 2020
TARGET_MAX_FISCAL_YEAR = 2025


## 3. EDGAR access sanity check

This step verifies that EDGAR access and local storage are working. It is disabled by default because it calls EDGAR.


In [4]:
if RUN_EDGAR_SANITY_CHECK:
    use_local_storage(True)

    try:
        tickers = get_company_tickers()
        df_tickers = tickers.to_pandas() if hasattr(tickers, "to_pandas") else tickers

        print("Success. Local EDGAR storage connected.")
        print(f"Total tickers: {len(df_tickers)}")
        display(df_tickers.head())
        print(df_tickers.columns)

    except Exception as e:
        print(f"EDGAR sanity check failed: {e}")
else:
    print("EDGAR sanity check skipped. Set RUN_EDGAR_SANITY_CHECK=True to execute.")


EDGAR sanity check skipped. Set RUN_EDGAR_SANITY_CHECK=True to execute.


## 4. Optional company-level probe

This small probe was used to understand what the `Company` object exposes for selected tickers.


In [5]:
if RUN_EDGAR_SANITY_CHECK:
    from edgar import Company

    for t in ["A", "WMT", "JPM"]:
        try:
            c = Company(t)
            facts = c.get_facts()

            assets = facts.get_total_assets()

            print(f"Ticker: {t:4} | SIC: {c.sic} | Assets: ${assets:,.0f}")

        except Exception as e:
            print(f"Error for {t}: {e}")
else:
    print("Company-level EDGAR probe skipped.")


Company-level EDGAR probe skipped.


## 5. Build the fundamental ticker universe

This step creates the first ticker / CIK / SIC checkpoint. It is resumable and writes to `data/raw/fundamental_universe.csv`.

This is an EDGAR-calling step and is disabled by default.


In [6]:
if RUN_TICKER_UNIVERSE_BUILD:
    use_local_storage(True)

    df = get_company_tickers()
    output_file = FUNDAMENTAL_UNIVERSE_PATH

    # Resume if file already exists.
    if output_file.exists():
        final_df = pd.read_csv(output_file)
        completed = set(final_df["ticker"].tolist())
        print(f"Resuming... already completed: {len(completed)}")
    else:
        final_df = pd.DataFrame()
        completed = set()

    rows = []

    for ticker in tqdm(df["ticker"].tolist()):
        if ticker in completed:
            continue

        try:
            c = Company(ticker)
            facts = c.get_facts()

            rows.append({
                "ticker": ticker,
                "cik": c.cik,
                "name": c.name,
                "sic": c.sic,
                "sic_description": getattr(c, "sic_description", None),
                "total_assets": facts.get_total_assets(),
                "total_liabilities": facts.get_total_liabilities(),
                "revenue": facts.get_revenue(),
                "net_income": facts.get_net_income(),
            })

            # Save every 100 rows.
            if len(rows) >= 100:
                temp_df = pd.DataFrame(rows)
                final_df = pd.concat([final_df, temp_df], ignore_index=True)
                final_df.to_csv(output_file, index=False)
                rows = []

        except Exception:
            continue

    # Final save.
    if rows:
        temp_df = pd.DataFrame(rows)
        final_df = pd.concat([final_df, temp_df], ignore_index=True)
        final_df.to_csv(output_file, index=False)

    print("Done.")
    print(f"Total saved: {len(final_df)}")
else:
    print("Ticker universe build skipped. Set RUN_TICKER_UNIVERSE_BUILD=True to execute.")


Ticker universe build skipped. Set RUN_TICKER_UNIVERSE_BUILD=True to execute.


## 6. Download the SEC SIC table

This optional step retrieves the SEC SIC classification table. It is separated from the ticker universe step so the source of the industry labels is clear.


In [7]:
if RUN_SIC_TABLE_DOWNLOAD:
    url = "https://www.sec.gov/corpfin/division-of-corporation-finance-standard-industrial-classification-sic-code-list"

    headers = {
        "User-Agent": "your_email@domain.com",  # Replace with your contact email before running.
        "Accept-Language": "en-US,en;q=0.9",
        "Accept-Encoding": "gzip, deflate, br",
        "Host": "www.sec.gov",
    }

    response = requests.get(url, headers=headers)
    print(response.status_code)

    html_io = StringIO(response.text)
    tables = pd.read_html(html_io)

    sic_df = tables[0]
    sic_df = sic_df.rename(columns={"SIC Code": "sic"})
    sic_df.info()
else:
    print("SIC table download skipped. Set RUN_SIC_TABLE_DOWNLOAD=True to execute.")


SIC table download skipped. Set RUN_SIC_TABLE_DOWNLOAD=True to execute.


## 7. Enrich the universe with SIC industry titles

This step merges the fundamental universe with SIC industry titles and stores an intermediate checkpoint.


In [8]:
if RUN_SIC_ENRICHMENT:
    fundamental_df = pd.read_csv(FUNDAMENTAL_UNIVERSE_PATH)
    fundamental_df = fundamental_df.dropna(subset=["sic", "ticker"])

    fundamental_df = fundamental_df.merge(
        sic_df,
        how="left",
        on="sic",
    )

    fundamental_df = fundamental_df.drop(columns=["sic_description"])
    fundamental_df.to_csv(TICKER_CIK_SIC_PATH, index=False)

    display(fundamental_df.head())
else:
    print("SIC enrichment skipped. Set RUN_SIC_ENRICHMENT=True to execute.")


SIC enrichment skipped. Set RUN_SIC_ENRICHMENT=True to execute.


## 8. Load the enriched ticker universe checkpoint

From this point onward, the notebook uses the saved universe checkpoint instead of calling EDGAR again.


In [9]:
fundamental_df = pd.read_csv(TICKER_CIK_SIC_PATH)
fundamental_df


FileNotFoundError: [Errno 2] No such file or directory: 'D:\\users\\kamen.dimitrov\\desktop\\SOFTUNI\\AI_and_ML_upskill_program\\machine_learning\\08_final_project_3\\data\\interim\\fundamental_universe_ticker_cik_sic.csv'

## 9. Audit SIC coverage

These cells check whether the universe has usable SIC and industry coverage before downloading raw EDGAR facts.


In [ ]:
fundamental_df["sic"].value_counts()


In [ ]:
fundamental_df["Industry Title"].value_counts()


In [ ]:
sic_audit = (
    fundamental_df
    .groupby(["sic", "Industry Title"])
    .agg(
        company_count=("ticker", "count"),
        sample_tickers=("ticker", lambda x: ", ".join(x.astype(str).head(10))),
        sample_names=("name", lambda x: " | ".join(x.astype(str).head(5))),
    )
    .reset_index()
    .sort_values("company_count", ascending=False)
)

sic_audit


In [ ]:
fundamental_df[fundamental_df["ticker"] == "ADP"]


## 10. Add broad SIC major-division buckets

The SEC SIC titles are detailed. This mapping creates broader industry groups for downstream credit-risk analysis.


In [ ]:
sic_major_division_map = {
    range(100, 1000): "Agriculture",
    range(1000, 1500): "Mining / Energy",
    range(1500, 1800): "Construction",
    range(2000, 4000): "Manufacturing / Industrials",
    range(4000, 5000): "Transportation / Utilities",
    range(5000, 6000): "Wholesale / Retail",
    range(6000, 6800): "Finance / Insurance / Real Estate",
    range(7000, 9000): "Services",
    range(9100, 9730): "Public Administration",
}


In [ ]:
def map_sic_major_division(sic):
    if pd.isna(sic):
        return "Unknown"

    sic = int(sic)

    for sic_range, industry in sic_major_division_map.items():
        if sic in sic_range:
            return industry

    return "Other"

if RUN_SIC_ENRICHMENT:
    fundamental_df = fundamental_df.copy()
    fundamental_df["industry"] = fundamental_df["sic"].apply(map_sic_major_division)
    fundamental_df.to_csv(TICKER_SIC_INDUSTRY_PATH, index=False)
    display(fundamental_df["industry"].value_counts())
else:
    print("Major-division mapping write skipped. Set RUN_SIC_ENRICHMENT=True to execute.")


## 11. Load the final universe used for EDGAR fact extraction

This file is the final company universe checkpoint used by the raw fact download step.


In [ ]:
fundamental_df = pd.read_csv(TICKER_SIC_INDUSTRY_PATH)
fundamental_df.head()


## 12. Download raw EDGAR financial facts

This is the high-risk, high-volume step. It is resumable and writes to `data/raw/raw_financial_facts.csv`.

Safety rules:

- The step is skipped unless `RUN_RAW_EDGAR_FACT_DOWNLOAD=True`.
- Full-universe downloads are blocked unless `CONFIRM_FULL_EDGAR_DOWNLOAD=True`.
- Use `MAX_TICKERS_FOR_DOWNLOAD` for small validation runs.


In [ ]:
if RUN_RAW_EDGAR_FACT_DOWNLOAD:
    if MAX_TICKERS_FOR_DOWNLOAD is None and not CONFIRM_FULL_EDGAR_DOWNLOAD:
        raise RuntimeError(
            "Full EDGAR download is disabled by default. "
            "Set CONFIRM_FULL_EDGAR_DOWNLOAD=True if you intentionally want to run all tickers."
        )

    use_local_storage(True)

    output_file = RAW_FACTS_CSV_PATH

    # Use your cleaned universe.
    tickers = fundamental_df["ticker"].dropna().unique().tolist()

    if MAX_TICKERS_FOR_DOWNLOAD is not None:
        tickers = tickers[:MAX_TICKERS_FOR_DOWNLOAD]
        print(f"Test run enabled. Downloading only {len(tickers)} tickers.")
    else:
        print(f"Full run enabled. Downloading {len(tickers)} tickers.")

    # Resume logic.
    if output_file.exists():
        existing = pd.read_csv(output_file)
        completed = set(existing["ticker"].unique())
        print(f"Resuming. Completed tickers: {len(completed)}")
    else:
        existing = pd.DataFrame()
        completed = set()

    rows = []

    for ticker in tqdm(tickers):
        if ticker in completed:
            continue

        try:
            c = Company(ticker)
            facts = c.get_facts()
            df = facts.to_dataframe()

            df["ticker"] = ticker
            df["cik"] = c.cik
            df["company_name"] = c.name
            df["sic"] = c.sic

            # Keep only accounting facts with numeric values.
            df = df[df["numeric_value"].notna()].copy()

            # Optional: keep only annual + quarterly periods.
            df = df[df["fiscal_period"].isin(["FY", "Q1", "Q2", "Q3", "Q4"])]

            rows.append(df)

            # Checkpoint every 25 companies.
            if len(rows) >= 25:
                batch = pd.concat(rows, ignore_index=True)

                if output_file.exists():
                    batch.to_csv(output_file, mode="a", header=False, index=False)
                else:
                    batch.to_csv(output_file, index=False)

                rows = []

        except Exception as e:
            print(f"Error for {ticker}: {e}")
            continue

    # Final save.
    if rows:
        batch = pd.concat(rows, ignore_index=True)

        if output_file.exists():
            batch.to_csv(output_file, mode="a", header=False, index=False)
        else:
            batch.to_csv(output_file, index=False)

    print("Done.")
else:
    print("Raw EDGAR financial fact download skipped. Set RUN_RAW_EDGAR_FACT_DOWNLOAD=True to execute.")


## 13. Convert raw CSV to parquet chunks

The raw CSV can become very large. Parquet chunks are faster and cheaper to query with DuckDB.


In [ ]:
if RUN_CSV_TO_PARQUET:
    csv_file = RAW_FACTS_CSV_PATH
    output_folder = PARQUET_DIR

    output_folder.mkdir(parents=True, exist_ok=True)

    chunksize = 250_000

    print("Starting CSV → Parquet chunk conversion...")

    for i, chunk in enumerate(pd.read_csv(csv_file, chunksize=chunksize)):
        output_path = output_folder / f"part_{i:05d}.parquet"

        # Save chunk as parquet.
        chunk.to_parquet(output_path, index=False)

        print(f"Saved chunk {i} → {output_path} ({len(chunk):,} rows)")

    print("Done. All chunks converted.")
else:
    print("CSV to parquet conversion skipped. Set RUN_CSV_TO_PARQUET=True to execute.")


## 14. Inspect parquet footprint

This confirms how many parquet chunks exist and the total local storage footprint.


In [ ]:
files = glob.glob(str(PARQUET_DIR / "*.parquet"))

print("Parquet files:", len(files))

total_size_gb = sum(os.path.getsize(f) for f in files) / 1e9
print("Total parquet size:", round(total_size_gb, 2), "GB")


## 15. Create a DuckDB view over parquet files

DuckDB allows the notebook to query the parquet folder without loading the entire dataset into pandas memory.


In [ ]:
con = duckdb.connect(str(DUCKDB_PATH))

con.execute(f"""
CREATE OR REPLACE VIEW raw_facts AS
SELECT *
FROM read_parquet('{str(PARQUET_DIR / "*.parquet")}')
""")

summary = con.execute("""
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT ticker) AS tickers,
    COUNT(DISTINCT concept) AS concepts,
    MIN(fiscal_year) AS min_year,
    MAX(fiscal_year) AS max_year
FROM raw_facts
""").df()

summary


In [ ]:
con.execute("""
SELECT fiscal_year, COUNT(*) AS n
FROM raw_facts
GROUP BY fiscal_year
ORDER BY fiscal_year
LIMIT 30
""").df()


## 16. Build the model base table

This table filters the raw facts to the fiscal-year window used for model training.


In [ ]:
con.execute(f"""
CREATE OR REPLACE TABLE credit_model_base AS
SELECT *
FROM raw_facts
WHERE fiscal_year BETWEEN {TARGET_MIN_FISCAL_YEAR} AND {TARGET_MAX_FISCAL_YEAR}
""")


In [ ]:
con.execute("""
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT ticker) AS tickers,
    COUNT(DISTINCT concept) AS concepts
FROM credit_model_base
""").df()


## 17. Concept coverage audit

This section identifies which XBRL concepts have broad ticker coverage. These concepts become candidates for downstream feature engineering.


In [ ]:
concept_counts = con.execute("""
SELECT
    concept,
    label,
    COUNT(*) AS row_count,
    COUNT(DISTINCT ticker) AS ticker_coverage
FROM credit_model_base
GROUP BY concept, label
ORDER BY ticker_coverage DESC
""").df()


In [ ]:
concept_counts


In [ ]:
top_concepts = con.execute("""
SELECT
    concept,
    label,
    COUNT(*) AS row_count,
    COUNT(DISTINCT ticker) AS ticker_coverage
FROM credit_model_base
GROUP BY concept, label
ORDER BY ticker_coverage DESC
LIMIT 200
""").df()

top_concepts


In [ ]:
target_concepts = [
    "us-gaap:Assets",
    "us-gaap:AssetsCurrent",
    "us-gaap:Liabilities",
    "us-gaap:LiabilitiesCurrent",
    "us-gaap:CashAndCashEquivalentsAtCarryingValue",
    "us-gaap:AccountsReceivableNetCurrent",
    "us-gaap:InventoryNet",
    "us-gaap:PropertyPlantAndEquipmentNet",
    "us-gaap:Goodwill",
    "us-gaap:LongTermDebt",
    "us-gaap:LongTermDebtCurrent",
    "us-gaap:StockholdersEquity",
    "us-gaap:RetainedEarningsAccumulatedDeficit",

    "us-gaap:Revenues",
    "us-gaap:GrossProfit",
    "us-gaap:OperatingIncomeLoss",
    "us-gaap:NetIncomeLoss",
    "us-gaap:InterestExpense",
    "us-gaap:SellingGeneralAndAdministrativeExpense",
    "us-gaap:ResearchAndDevelopmentExpense",

    "us-gaap:NetCashProvidedByUsedInOperatingActivities",
    "us-gaap:PaymentsToAcquirePropertyPlantAndEquipment",
]

target_concepts


## 18. Incremental refresh strategy: new tickers and new fiscal years

The dataset needs two ongoing refresh checks:

1. **New ticker detection:** EDGAR may contain tickers that were not present in the last local universe checkpoint.
2. **New annual filing detection:** existing companies file new annual reports over time, so the latest fiscal year coverage can become stale.

This section is an audit scaffold. It does not download anything unless the relevant download flags are enabled elsewhere.


In [ ]:
MANIFEST_PATH = PROCESSED_DIR / "edgar_download_manifest.csv"
SNAPSHOT_DATE = date.today().isoformat()
LATEST_TICKER_SNAPSHOT_PATH = RAW_DIR / f"company_tickers_{SNAPSHOT_DATE}.csv"

if RUN_INCREMENTAL_REFRESH_AUDIT:
    use_local_storage(True)

    latest_tickers = get_company_tickers()
    latest_tickers = latest_tickers.to_pandas() if hasattr(latest_tickers, "to_pandas") else latest_tickers
    latest_tickers.to_csv(LATEST_TICKER_SNAPSHOT_PATH, index=False)

    current_universe = pd.read_csv(TICKER_SIC_INDUSTRY_PATH)

    existing_tickers = set(current_universe["ticker"].dropna().astype(str))
    latest_ticker_set = set(latest_tickers["ticker"].dropna().astype(str))

    new_tickers = sorted(latest_ticker_set - existing_tickers)
    removed_or_changed_tickers = sorted(existing_tickers - latest_ticker_set)

    print("New tickers:", len(new_tickers))
    print("Removed / changed tickers:", len(removed_or_changed_tickers))
else:
    print("Incremental ticker refresh audit skipped. Set RUN_INCREMENTAL_REFRESH_AUDIT=True to execute.")


In [ ]:
# Build or refresh a local coverage manifest from the parquet-backed DuckDB view.
# This does not call EDGAR. It summarizes what is already stored locally.

manifest_query = """
SELECT
    ticker,
    cik,
    company_name,
    sic,
    MIN(fiscal_year) AS min_fiscal_year,
    MAX(fiscal_year) AS max_fiscal_year,
    COUNT(*) AS row_count
FROM raw_facts
GROUP BY ticker, cik, company_name, sic
ORDER BY ticker
"""

if "con" in globals():
    manifest_df = con.execute(manifest_query).df()
    manifest_df["last_audited_at"] = date.today().isoformat()
    manifest_df.to_csv(MANIFEST_PATH, index=False)
    display(manifest_df.head())
else:
    print("DuckDB connection not found. Run the DuckDB view section first.")


In [ ]:
# Identify stale tickers where local data does not yet reach the target maximum fiscal year.

if MANIFEST_PATH.exists():
    manifest = pd.read_csv(MANIFEST_PATH)

    stale_tickers = manifest.loc[
        manifest["max_fiscal_year"] < TARGET_MAX_FISCAL_YEAR,
        "ticker",
    ].dropna().astype(str).tolist()

    print("Stale tickers needing refresh:", len(stale_tickers))
else:
    stale_tickers = []
    print("No manifest found yet. Run the manifest cell after creating the DuckDB raw_facts view.")


## 19. Operational notes

Suggested Git hygiene:

- Keep the notebook under `notebooks/`.
- Keep generated EDGAR files under `data/`.
- Commit `data/README.md`, but do not commit large raw CSV, parquet, or DuckDB files.
- Refresh the universe periodically to detect new tickers.
- Refresh recent fiscal years conservatively, because companies may amend or restate filings.
